# 05 — Cross-Algorithm Comparison

Compares Q-Learning, Dyna-Q, Dyna-Q Tuned, and A2C across all environments and decay strategies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 110

ROOT      = os.path.abspath('..')
DATA_DIR  = os.path.join(ROOT, 'data')
PLOTS_DIR = os.path.join(ROOT, 'plots')

ENVS = [('empty_5x5','Empty 5×5'),('empty_8x8','Empty 8×8'),('empty_16x16','Empty 16×16'),
        ('doorkey_5x5','DoorKey 5×5'),('doorkey_8x8','DoorKey 8×8'),('doorkey_16x16','DoorKey 16×16')]
ENVS_EMPTY = ENVS[:3]; ENVS_DOORKEY = ENVS[3:]
DECAYS = ['linear','exp','rbed_linear','rbed_exp']
DECAY_LABELS = {'linear':'Linear Decay','exp':'Exponential Decay','rbed_linear':'RBED + Linear','rbed_exp':'RBED + Exponential'}
DECAY_COLORS = {'linear':'#2266CC','exp':'#22AA55','rbed_linear':'#CC4422','rbed_exp':'#AA22AA'}
DECAY_STYLES = {'linear':'-','exp':'--','rbed_linear':':','rbed_exp':'-.'}
ALGO_LIST = ['qlearning','dynaq','dynaq_tuned','a2c_td']
ALGO_LABELS = {'qlearning':'Q-Learning','dynaq':'Dyna-Q (n=5)','dynaq_tuned':'Dyna-Q Tuned (n=20)','a2c_td':'A2C (TD)'}
ALGO_COLORS = {'qlearning':'royalblue','dynaq':'green','dynaq_tuned':'teal','a2c_td':'tomato'}
ALGO_STYLES = {'qlearning':'-','dynaq':'--','dynaq_tuned':'-.','a2c_td':':'}

def load(algo, env, decay):
    folder = os.path.join(DATA_DIR, algo)
    metrics = ['rewards','steps','epsilons','success','avg_q']
    paths = {m: os.path.join(folder, f'{algo}__{env}__{decay}__{m}.npy') for m in metrics}
    if not all(os.path.exists(p) for p in paths.values()): return None
    return {m: np.load(paths[m]) for m in metrics}

def smooth(x, w=100):
    if len(x)<w: return x
    return np.convolve(x, np.ones(w)/w, mode='valid')

def summary(st, last_n=500):
    if st is None: return None
    r, s, steps = st['rewards'], st['success'], st['steps']
    first = next((i for i,v in enumerate(s) if v>0), None)
    peak  = float(np.max(np.convolve(s,np.ones(100)/100,'valid'))) if len(s)>=100 else float(np.mean(s))
    return {'sr':float(np.mean(s[-last_n:])),'sr100':float(np.mean(s[-100:])),'avg_r':float(np.mean(r[-last_n:])),
            'avg_steps':float(np.mean(steps[-last_n:])),'first':first,'peak_sr':peak}

def savefig(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    print(f'Saved: {path}')

OUT = os.path.join(PLOTS_DIR, 'comparison')
os.makedirs(OUT, exist_ok=True)
print("✅ Setup complete")

## 1. All Algorithms — Per Environment × Per Decay

In [ ]:
for env_id, env_label in ENVS:
    for decay in DECAYS:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle(f'Algorithm Comparison — {env_label}  [{DECAY_LABELS[decay]}]', fontsize=13, fontweight='bold')
        ax_r, ax_sr, ax_eps = axes
        for algo in ALGO_LIST:
            st = load(algo, env_id, decay)
            if st is None: continue
            c, ls, lbl = ALGO_COLORS[algo], ALGO_STYLES[algo], ALGO_LABELS[algo]
            ax_r.plot(smooth(st['rewards']),  color=c, ls=ls, lw=1.8, label=lbl)
            ax_sr.plot(smooth(st['success']), color=c, ls=ls, lw=1.8, label=lbl)
            ax_eps.plot(st['epsilons'],        color=c, ls=ls, lw=1.2, alpha=0.85, label=lbl)
        ax_r.axhline(0, color='k', ls='--', lw=0.6, alpha=0.4)
        for ax, ylabel, title in [(ax_r,'Total Reward (avg 100)','Training Curve'),
                                   (ax_sr,'Success Rate (avg 100)','Success Rate'),
                                   (ax_eps,'Epsilon ε','Exploration Rate')]:
            ax.set_xlabel('Episode'); ax.set_ylabel(ylabel)
            ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax_sr.set_ylim(-0.05, 1.05); ax_eps.set_ylim(0, 1.05)
        plt.tight_layout()
        savefig(fig, f'{OUT}/compare__{env_id}__{decay}.png')
        plt.show()

## 2. Heatmap — All Algorithms × All Environments (per decay)

In [ ]:
for decay in DECAYS:
    sr_mat = np.full((len(ALGO_LIST), len(ENVS)), np.nan)
    r_mat  = np.full((len(ALGO_LIST), len(ENVS)), np.nan)
    for i, algo in enumerate(ALGO_LIST):
        for j, (env_id, _) in enumerate(ENVS):
            sm = summary(load(algo, env_id, decay))
            if sm: sr_mat[i,j]=sm['sr']; r_mat[i,j]=sm['avg_r']
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    fig.suptitle(f'All Algorithms — {DECAY_LABELS[decay]}', fontsize=13, fontweight='bold')
    for ax, mat, title, fmt in [(axes[0],sr_mat,'Success Rate','sr'),(axes[1],r_mat,'Avg Reward','r')]:
        vmin=np.nanmin(mat); vmax=np.nanmax(mat)
        im = ax.imshow(mat, cmap='RdYlGn', aspect='auto', vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(ENVS))); ax.set_xticklabels([e[1] for e in ENVS], rotation=30, ha='right', fontsize=8)
        ax.set_yticks(range(len(ALGO_LIST))); ax.set_yticklabels([ALGO_LABELS[a] for a in ALGO_LIST], fontsize=9)
        ax.axvline(2.5, color='white', lw=2.5); ax.set_title(title, fontsize=11, fontweight='bold')
        for i in range(len(ALGO_LIST)):
            for j in range(len(ENVS)):
                v=mat[i,j]
                if np.isnan(v): continue
                txt=f'{v:.0%}' if fmt=='sr' else f'{v:.2f}'
                nv=(v-vmin)/(vmax-vmin+1e-9)
                ax.text(j,i,txt,ha='center',va='center',fontsize=8,fontweight='bold',color='white' if nv<0.35 else 'black')
        plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    plt.tight_layout()
    savefig(fig, f'{OUT}/heatmap__{decay}.png')
    plt.show()

## 3. Scalability — Success Rate vs Environment Size (Linear Decay)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Scalability: Success Rate vs Environment Size (Linear Decay)', fontsize=13, fontweight='bold')
sizes = ['5×5', '8×8', '16×16']
for col, (group, env_group) in enumerate([('Empty Environments', ENVS_EMPTY),('DoorKey Environments', ENVS_DOORKEY)]):
    ax = axes[col]
    for algo in ALGO_LIST:
        srs = [summary(load(algo,e[0],'linear'))['sr'] if summary(load(algo,e[0],'linear')) else 0 for e in env_group]
        ax.plot(sizes, srs, color=ALGO_COLORS[algo], ls=ALGO_STYLES[algo],
                lw=2.2, marker='o', ms=9, label=ALGO_LABELS[algo])
    ax.set_title(group); ax.set_xlabel('Environment Size')
    ax.set_ylabel('Success Rate (last 500 eps)'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3); ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
savefig(fig, f'{OUT}/scaling_analysis.png')
plt.show()

## 4. Best Decay Strategy per Algorithm × Environment

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Best Decay Strategy — All Algorithms & Environments', fontsize=13, fontweight='bold')
for idx, (env_id, env_label) in enumerate(ENVS):
    ax = axes[idx//3, idx%3]
    ax.set_title(env_label, fontsize=10, fontweight='bold')
    x = np.arange(len(ALGO_LIST))
    width = 0.2
    for di, decay in enumerate(DECAYS):
        srs = [summary(load(a,env_id,decay))['sr'] if summary(load(a,env_id,decay)) else 0 for a in ALGO_LIST]
        ax.bar(x + di*width, srs, width, label=DECAY_LABELS[decay],
               color=DECAY_COLORS[decay], alpha=0.85)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels([ALGO_LABELS[a] for a in ALGO_LIST], fontsize=7, rotation=15)
    ax.set_ylabel('Success Rate'); ax.set_ylim(0, 1.15)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
savefig(fig, f'{OUT}/winner_chart.png')
plt.show()

## 5. First Success Episode — Sample Efficiency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Sample Efficiency: First Success Episode (Linear Decay)', fontsize=13, fontweight='bold')
for col, (group, env_group) in enumerate([('Empty', ENVS_EMPTY),('DoorKey', ENVS_DOORKEY)]):
    ax = axes[col]
    x = np.arange(len(env_group))
    width = 0.2
    for ai, algo in enumerate(ALGO_LIST):
        firsts = []
        for env_id, _ in env_group:
            sm = summary(load(algo, env_id, 'linear'))
            firsts.append(sm['first'] if sm and sm['first'] else 5000)
        ax.bar(x + ai*width, firsts, width, label=ALGO_LABELS[algo], color=ALGO_COLORS[algo], alpha=0.85)
    ax.set_xticks(x + width*1.5); ax.set_xticklabels([e[1] for e in env_group], fontsize=9)
    ax.set_title(f'{group} Environments'); ax.set_ylabel('First Success Episode (lower = better)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
savefig(fig, f'{OUT}/first_success.png')
plt.show()

## 6. Full Comparison Table

In [ ]:
print(f"{'Algo':<18} {'Env':<16} {'Decay':<22} {'SR%':>6} {'AvgR':>7} {'First':>8} {'PeakSR':>8}")
print("="*85)
for algo in ALGO_LIST:
    for env_id, env_label in ENVS:
        for decay in DECAYS:
            sm = summary(load(algo, env_id, decay))
            if not sm: continue
            fe = f"ep{sm['first']}" if sm['first'] else 'Never'
            print(f"  {ALGO_LABELS[algo]:<16} {env_label:<16} {DECAY_LABELS[decay]:<22} {sm['sr']:>5.0%} {sm['avg_r']:>7.3f} {fe:>8} {sm['peak_sr']:>7.0%}")
    print()